# 🌪️ Modélisation Temporelle & Météo (RNN / LSTM) — Prédiction de la Propagation
**Responsable :** Moundir — *Responsable Modélisation Temporelle & Météo (RNN/LSTM) & Support Data*  
**Architecture choisie :** `FirePropagationLSTM` (Bi-LSTM Bidirectionnel avec Pooling Temporel & Têtes Vitesse/Direction)

---
### Récapitulatif des Missions Couvertes :
1. **Tâche 1 (Traitement des données météorologiques & séries temporelles)** : Analyse et transformation des variables atmosphériques (vent, humidité relative, température), encodage cyclique des directions angulaires en composantes `(sin, cos)`, et structuration en séquences temporelles glissantes.
2. **Tâche 2 (Entraînement du modèle RNN/LSTM)** : Entraînement avec fonction de perte hybride (Smooth L1 + Cosine Vector Loss) pour prédire conjointement la vitesse de propagation ($v \ge 0$ km/h) et la direction cardinale.
3. **Tâche 3 (Sauvegarde `.pkl` & Fonction `predict_propagation(location, wind, humidity)`)** : Sérialisation dans `models/fire_propagation_lstm.pkl`, et fourniture de la fonction d'inférence officielle retournant la direction et la vitesse de propagation.
4. **Mission Secondaire (Support Data)** : Préparation des séries temporelles synchronisées avec la localisation CNN de Salim et le Système d'Alerte (`main.py` / Streamlit).

In [ ]:
import numpy as np
import pandas as pd
import torch

from temporal_preprocessing import (
    generate_synthetic_propagation_sequences,
    create_sliding_sequences,
    prepare_datasets_for_training,
    direction_to_angle,
    angle_to_cardinal,
)
from model_lstm import FirePropagationLSTM, train_propagation_lstm, load_lstm_model
from predict_propagation import predict_propagation

print("PyTorch Version:", torch.__version__)
print("Disponibilité CUDA:", torch.cuda.is_available())

## 1. Tâche 1 : Traitement Météorologique & Préparation des Séries Temporelles
Nous analysons le dataset météo Kaggle `forestfires_kaggle.csv` et construisons des séries temporelles glissantes de 4 pas horaires avec feature engineering (sin/cos du vent, localisation et inertie du feu).

In [ ]:
df_series = generate_synthetic_propagation_sequences(
    base_dataset_csv="dataset/forestfires_kaggle.csv",
    num_episodes=50,
    episode_length=12,
    seed=42,
)

print("Forme du dataset temporel :", df_series.shape)
display(df_series[["episode_id", "hour_step", "temperature_c", "humidity_pct", "wind_speed_kmh", "propagation_direction_deg", "target_speed_kmh"]].head(6))

## 2. Tâche 2 : Entraînement du Modèle RNN/LSTM (`FirePropagationLSTM`)
Entraînement avec optimiseur AdamW, régularisation CosineAnnealing, et calcul des métriques de validation (MAE Vitesse en km/h et Erreur Angulaire en degrés).

In [ ]:
model, metrics = train_propagation_lstm(
    csv_source="dataset/forestfires_kaggle.csv",
    epochs=25,
    batch_size=32,
    lr=1.5e-3,
    save_path_pkl="models/fire_propagation_lstm.pkl",
    metrics_json_path="outputs/lstm_training_metrics.json",
    seed=42,
)

print(f"\nRésultats finaux : MAE Vitesse = {metrics['val_speed_mae_kmh']} km/h | Erreur Angulaire = {metrics['val_dir_angular_error_deg']}°")

## 3. Tâche 3 : Fonction Officielle `predict_propagation(location, wind, humidity)`
Validation sur différents scénarios opérationnels (vent fort sec, brise modérée).

In [ ]:
res1 = predict_propagation(
    location={"zone": "Nord-Est", "affected_area_pct": 7.5, "normalized_coords": (0.75, 0.25)},
    wind={"speed": 42.0, "direction": "Sud-Ouest"},
    humidity=20.0,
    temperature=35.0,
)

print("=== SCÉNARIO 1 : VENT FORT SUD-OUEST ===")
print("Direction prédite :", res1["propagation_direction"])
print("Vitesse prédite   :", res1["propagation_speed"], "km/h")
print("Synthèse          :", res1["summary"])

In [ ]:
res2 = predict_propagation(
    location="Centre",
    wind=25.0,
    humidity=35.0,
)

print("=== SCÉNARIO 2 : ENTRÉE MINIMALE ===")
print("Direction :", res2["direction"])
print("Vitesse   :", res2["speed"], "km/h")
print("Détail    :", res2["summary"])

## 4. Pipeline CNN + LSTM
Démonstration de la liaison directe entre la détection spatiale (Salim) et la propagation temporelle (Moundir).

In [ ]:
from predict import predict

sample_image = "dataset/satellite_samples/images/fire/sat_fire_0001.png"
cnn_output = predict(sample_image)

print("--- 1. RÉSULTAT DU CNN (SALIM) ---")
print("Feu détecté :", cnn_output["fire_detected"])
print("Confiance   :", cnn_output["confidence"])
print("Zone        :", cnn_output["location"]["zone"])

if cnn_output["has_fire"]:
    lstm_output = predict_propagation(
        location=cnn_output["location"],
        wind={"speed": 32.0, "direction": "Nord-Est"},
        humidity=28.0,
        temperature=31.0,
    )
    print("\n--- 2. RÉSULTAT DU LSTM (MOUNDIR) ---")
    print("Direction de propagation :", lstm_output["propagation_direction"])
    print("Vitesse de propagation   :", lstm_output["propagation_speed"], "km/h")
    print("Synthèse Globale         :", lstm_output["summary"])